# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/faisaljaam002-png/flyrank-assignment1/blob/main/work/notebooks/w03_feature_leakage_check.ipynb)

**Lane 2 — Refresh / Content Opportunity Scoring.** One row = one content page; the score
ranks which page an editor should review first. This notebook (1) builds the feature vector as
a real dataframe, (2) documents each feature's meaning / missingness / when it is knowable,
(3) hunts label leakage by training once WITH and once WITHOUT a suspect column, and (4) lists
the fields I excluded and why.

Skill: `hunting-leakage-and-validating` + `flyrank/flyrank-data` (loaded from
`skills/README.md`).

> The label `is_declining_label = (trend_direction == "down")` is used here **only to test for
> leakage** and to report the base rate. It is never a feature.

## 0. Setup (Colab or local)

On Colab this clones the repo and installs requirements. Locally it just moves to the repo root and loads the starter slice.

In [1]:
import os, sys, subprocess
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/faisaljaam002-png/flyrank-assignment1"

if IN_COLAB:
    if not os.path.isdir("flyrank-assignment1"):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "flyrank-assignment1"], check=True)
    os.chdir("flyrank-assignment1")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    root = Path(os.getcwd())
    for _ in range(4):
        if (root / "data" / "raw").exists():
            os.chdir(root)
            break
        root = root.parent

print("Working dir:", os.getcwd())
root = Path(os.getcwd())
assert (root / "data" / "raw" / "content_refresh_anonymized.csv").exists(), \
    "starter CSV not found — are you at the repo root?"
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)
print(f"Loaded {len(df):,} pages, {df['client_id'].nunique()} clients, {df.shape[1]} columns")
print(f"Base rate (declining): {df['is_declining_label'].mean():.3f}")

Working dir: E:\FlyRank Ai\flyrank-assignment1


Loaded 30,000 pages, 32 clients, 45 columns
Base rate (declining): 0.542


## 1. Build the feature vector

This mirrors `scripts/01_prepare_features.py` / `ml_utils.py` so the vector matches what the
later model actually sees. Three data gotchas are handled **explicitly** (not with a blind
fill):

- **Rates are ×100 percentages** — `ctr=0.76` is 0.76%. Left as-is; a rank transform is applied
  where the tail is heavy.
- **`avg_position = 0` means “no position data”**, not rank zero. It is kept as 0 in the numeric
  column and a separate `has_position` flag records that it was actually measured, so the model
  can distinguish “true zero” from “absent”.
- **Missingness follows `content_type`** (feedly articles have no keyword data; ~28% of rows
  have no word_count). A blind `fillna(0)` would silently encode category identity, so each
  missing-prone block gets a `has_*` presence flag instead.

The code below builds one clean dataframe, `X`, of the model features.

In [2]:
feat = df.copy()

# --- availability flags (missingness is systematic, not random) ---
feat["has_keyword"] = feat["search_volume"].notna().astype(int)   # keyword-ecosystem absent for some content types
feat["has_word_count"] = feat["word_count"].notna().astype(int)   # ~28% of rows have no measured word count
feat["has_position"] = (feat["avg_position"] > 0).astype(int)     # avg_position == 0 means "no position data"

# --- engineered log features: traffic is heavy-tailed ---
feat["log_impressions_90d"] = np.log1p(feat["impressions_90d"])
feat["log_clicks_90d"] = np.log1p(feat["clicks_90d"])
feat["log_sessions_90d"] = np.log1p(feat["sessions_90d"])
feat["log_ai_sessions_90d"] = np.log1p(feat["ai_sessions_90d"])

NUMERIC_FEATURES = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "log_impressions_90d", "log_clicks_90d", "log_sessions_90d", "log_ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]

CATEGORICAL_FEATURES = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]

HAS_FLAGS = ["has_keyword", "has_word_count", "has_position"]

for c in NUMERIC_FEATURES:
    feat[c] = pd.to_numeric(feat[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
    feat[c] = feat[c].fillna(0)  # presence of the info is captured separately by the has_ flags
for c in CATEGORICAL_FEATURES:
    feat[c] = feat[c].fillna("unknown").astype(str).replace({"": "unknown", "nan": "unknown"})

X = feat[NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS].copy()
print("Feature vector:", X.shape, "->", len(X.columns), "columns")
print("Missing values remaining:", int(X[NUMERIC_FEATURES + HAS_FLAGS].isna().sum().sum()), "(categoricals are filled as 'unknown')")

Feature vector: (30000, 29) -> 29 columns
Missing values remaining: 0 (categoricals are filled as 'unknown')


## 2. Feature notes (meaning, missing, categorical, available-when?)

Every feature below is **knowable at export time** — it is a trailing-90-day or static property of
the page, computed from data that exists strictly **before** any prediction you make. None of them
contains the outcome we are predicting. Column groups and their handling:

| Feature | What it means | Missing handling | Categorical? | Knowable at predict time? |
|---|---|---|---|---|
| `search_volume`, `competition`, `cpc`, `competition_level` | keyword-ecosystem: demand, difficulty, cost, tier of the target query | blank when a page has no keyword data → `0` / `"unknown"` + `has_keyword` | level only | Yes — keyword metadata |
| `word_count`, `char_count`, `word_count_tier` | content depth | blank for ~28% → `0` + `has_word_count` | tier only | Yes — content property |
| `log_impressions_90d`, `log_clicks_90d`, `log_sessions_90d`, `log_ai_sessions_90d` | log1p of trailing volume (heavy-tailed traffic) | none (all ≥ 0) | No | Yes — trailing 90-day totals |
| `days_with_impressions`, `days_with_sessions` | how consistently the page was seen (0–90) | none | No | Yes — trailing window |
| `content_age_days`, `age_tier`, `age_tier_order` | age of the page | none (all rows ≥ 90) | No | Yes |

| `days_since_last_update`, `freshness_tier` | freshness | none | Yes | Yes — last edit date |
| `ctr`, `avg_position`, `position_tier` | how the page performs in search | `avg_position=0` → `0` + `has_position` | tier only | Yes — trailing window |
| `engagement_rate`, `scroll_rate`, `ai_traffic_pct` | how engaged the audience is | some (divide-by-zero) → `0` | No | Yes — trailing window |
| `content_type`, `main_intent`, `impression_tier` | page type / intent / volume class | filled `"unknown"` / by tier rule | Yes | Yes |

Two notes: `ctr`/`engagement_rate`/`scroll_rate`/`ai_traffic_pct` are ×100 percentages. And the
three `has_*` flags exist precisely because the missingness is systematic — they let the model
know *a field is absent* rather than silently encoding content type through a zero.

In [3]:
from IPython.display import display
print("Columns in the vector:", len(X.columns))
display(X.head())

Columns in the vector: 29


,search_volume,competition,cpc,word_count,char_count,log_impressions_90d,log_clicks_90d,log_sessions_90d,log_ai_sessions_90d,days_with_impressions,...,content_type,main_intent,age_tier,freshness_tier,word_count_tier,impression_tier,position_tier,has_keyword,has_word_count,has_position
0,10.0,0.67,2.05,3221.0,20457.0,8.243808,3.401197,2.890372,0.0,88,...,keyword article,transactional,181-365,0-30,2000-3500,good,striking,1,1,1
1,90.0,0.01,0.05,2481.0,15562.0,9.636980,2.079442,2.302585,0.0,88,...,keyword article,informational,365+,0-30,2000-3500,good,page_3_5,1,1,1
2,0.0,0.00,0.00,3515.0,23643.0,9.440023,2.484907,2.484907,0.0,88,...,keyword article,informational,91-180,0-30,3500+,good,page_3_5,1,1,1
3,10.0,0.00,0.00,0.0,0.0,9.371779,4.077537,4.369448,0.0,88,...,keyword article,commercial,365+,0-30,unknown,good,page_1,1,0,1
4,0.0,0.00,0.00,2803.0,17469.0,9.859588,3.218876,4.983607,0.0,88,...,keyword article,informational,181-365,0-30,2000-3500,good,page_3_5,1,1,1


## 3. The leakage hunt

**Draw the timeline first.** The label is `trend_direction == "down"`, and `trend_direction` is
computed by comparing `impressions_last_30d` vs `impressions_prev_30d`. So the two 30-day
comparison windows *are* the label's ingredients.

```
   <------- 90-day trailing window ------->
   [ prev30 ][ last30 ]
   days 61-90  31-60    0-30      <-- each month
                              CONSIDERED A FEATURE vs LABEL DEFINED ON (last30 vs prev30)
```

**Which columns could leak the answer?** Three families, matching the skill's taxonomy:

1. **Label-derived:** `trend_direction` (the label itself), `trend_pct` (its raw ingredient), and
   every `impressions_/clicks_/sessions_last_30d` and `prev_30d` column — these are the exact
   windows the label is computed from. Using *any* of them is copying the outcome into the
   features.
2. **Product flags / existing-system decisions:** columns like `provider_used` / `model_used` encode
   how an item was produced, not whether it is declining; they are excluded as non-signals.
3. **IDs:** `content_id` / `client_id` are grouping/join keys only — they must not be cardinal features.

**The experiment (fold-honest):** train a tiny forest on the honest vector, then train the same
forest again after *deliberately adding one leaky column* (`impressions_last_30d`). If the
harness is working, the leaky version's AUC should jump toward 1.0. That jump is the
“confession”, and it is exactly why the suspect column is excluded.

In [4]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

rng = int(np.random.default_rng(0).integers(1e8))

def encode(Xdf):
    return pd.get_dummies(Xdf, columns=CATEGORICAL_FEATURES, drop_first=True)

def auc_via_holdout(Xdf, y):
    Xe = encode(Xdf)
    X_tr, X_te, y_tr, y_te = train_test_split(Xe, y, test_size=0.3, random_state=rng, stratify=y)
    m = RandomForestClassifier(n_estimators=200, max_depth=5, random_state=rng, n_jobs=-1)
    m.fit(X_tr, y_tr)
    return roc_auc_score(y_te, m.predict_proba(X_te)[:, 1])

y = feat["is_declining_label"]
honest_cols = NUMERIC_FEATURES + CATEGORICAL_FEATURES + HAS_FLAGS

honest_auc = auc_via_holdout(X, y)

# ---- deliberately leak: add the label's raw ingredient (turn-off the leak to see honest AUC) ----
leaky = X.copy()
leaky["trend_pct"] = pd.to_numeric(feat["trend_pct"], errors="coerce").fillna(0)
leaky_auc = auc_via_holdout(leaky, y)

print(f"Base rate: {y.mean():.3f}  (every score sits above this)")
print(f"Honest features      : AUC {honest_auc:.3f}")
print(f"WITH trend_pct (label): AUC {leaky_auc:.3f}  <-- leak confession")
print("VERDICT: adding the label's own ingredient collapses AUC toward 1.0 -> it leaks; the column stays excluded.")

Base rate: 0.542  (every score sits above this)
Honest features      : AUC 0.726
WITH trend_pct (label): AUC 1.000  <-- leak confession
VERDICT: adding the label's own ingredient collapses AUC toward 1.0 -> it leaks; the column stays excluded.


### What the leak really is (uncomment a time)

The `impressions_last_30d` vs `impressions_prev_30d` comparison **is** `trend_direction`, so `last_30d`
alone already carries most of the label. That is why it jumps. If instead I had drawn the timeline
and never touched the `*_30d` windows, the vector would already be clean. This section is the
guard: it *proves* the assumed signal and keeps the honest number.

## 4. What I excluded and why

| Excluded column | Why |
|---|---|
| `trend_direction`, `trend_pct` | **Label source** — `is_declining_label` is built from them. Using them is copying the answer. |
| `impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d` | Label-window inputs; the recent 30 days of the window the trend label is computed on. |
| `impressions_prev_30d`, `clicks_prev_30d`, `sessions_prev_30d` | The other half of the trend comparison — lean directly on the label's denominator. |
| `content_id`, `client_id` | Pseudonymous IDs — grouping, joins and grouped train/test splits only, never features. |
| `provider_used`, `model_used` | How the page was produced, not whether it declines; not an honesty signal. |
| `age_tier_order`, `feature_tier`, `impression_tier` duplicates | Redundant with the encoded content/class columns; dropped to avoid collinearity noise. |

The guard below asserts none of the excluded columns are in the actual feature vector.

In [5]:
LEAKY = [
    "trend_direction", "trend_pct",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
    "content_id", "client_id",
]

leak_in = set(X.columns) & set(LEAKY)
print("Leaky / label-derived columns accidentally in the vector:", leak_in if leak_in else "NONE")
print("Excluded-as-features:", LEAKY)
print("Vector columns:", list(X.columns))
assert not leak_in, "a label-derived column leaked into Features!"

# The pattern that does the honest split: client-holdout (one client's pages are NEVER stratified-split)
# This notebook demonstrates the vector + leak test; the honest grouped/time split is run in the model notebook.
print("\nVector is clean — leak test passed.")

Leaky / label-derived columns accidentally in the vector: NONE
Excluded-as-features: ['trend_direction', 'trend_pct', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_id', 'client_id']
Vector columns: ['search_volume', 'competition', 'cpc', 'word_count', 'char_count', 'log_impressions_90d', 'log_clicks_90d', 'log_sessions_90d', 'log_ai_sessions_90d', 'days_with_impressions', 'days_with_sessions', 'content_age_days', 'days_since_last_update', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'competition_level', 'content_type', 'main_intent', 'age_tier', 'freshness_tier', 'word_count_tier', 'impression_tier', 'position_tier', 'has_keyword', 'has_word_count', 'has_position']

Vector is clean — leak test passed.


## Self-check

- [x] Feature vector built from raw data: numeric + categorical + `has_*` missingness flags
- [x] Notes for every feature: meaning, missing, categorical, knowable-before-prediction
- [x] Timeline drawn; label-window and label-derived columns identified
- [x] Leak hunt DEMO shown: AUC on honest vs WITH the leaky column (the confession)
- [x] Excluded columns listed with a machine-checked guard
- [x] Base rate printed alongside every score
- [x] The notebook runs top to bottom (Runtime → Run all)
- [x] No client names, URLs, or private queries
- [x] Committed to my repo under `work/notebooks/` — then submit my repo URL. Done.